In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# NVIDIA(OpenAI 호환 API) 클라이언트 초기화
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

# gpt-5-nano 대신 NVIDIA gpt-oss-20b 호출 예시 (추론 없이 바로 답)
def ask_direct(question: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{"role": "user", "content": f"{question} 최종 답만 숫자로 출력하라."}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(ask_direct("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))
# → "213000"   ← 틀림! (정답은 213300)

213300


In [4]:
# OpenAI SDK (gpt-5-nano) CoT 적용 예시
def ask_cot(question: str) -> str:
    resp = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": f"{question} 생각 과정을 단계별로 작성한 뒤 마지막에 정답을 출력하라."}],
        temperature=0,
    )
    return resp.choices[0].message.content

print(ask_cot("이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"))

**단계별 계산 과정**

1. **개별 가격 확인**  
   - 이어버드 한 개의 가격: 79,000원

2. **구매한 개수**  
   - 3개를 구매

3. **쿠폰 적용 전 총액 계산**  
   \[
   \text{총액} = 79,000 \text{원} \times 3 = 237,000 \text{원}
   \]

4. **10% 쿠폰 할인액 계산**  
   \[
   \text{할인액} = 237,000 \text{원} \times 0.10 = 23,700 \text{원}
   \]

5. **최종 결제액 계산**  
   \[
   \text{결제액} = 237,000 \text{원} - 23,700 \text{원} = 213,300 \text{원}
   \]

---

**정답**  
최종 결제액은 **213,300원**입니다.


In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

def ask_direct(question: str) -> str:
    """직접 답변: 중간 과정 없이 최종 숫자만 시킨다 → 다단계 계산에서 자주 틀림."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 최종 숫자(원)만 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [8]:
def ask_cot(question: str) -> str:
    """CoT: 한 줄씩 풀이를 쓰게 한다.

    [왜] 모델은 앞서 쓴 자기 출력을 다시 입력으로 참고한다. 풀이를 글로 쓰게 하면
    그 풀이가 다음 토큰 생성의 '작업 공간(근거)'이 되어 마지막 답이 정확해진다.
    """
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"{question}\n"
                    "단계적으로 풀어라. 각 계산을 한 줄씩 쓰고, "
                    "맨 마지막 줄에 '정답: <숫자>' 형식으로 답하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [9]:
import re

def extract_number(text: str):
    """답변에서 마지막 숫자를 정수로 반환(콤마 제거)."""
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    if not nums:
        return None
    return int(nums[-1].replace(",", ""))

In [10]:
q = "이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"
print("[직접] ", ask_direct(q))
print("[CoT]\n", ask_cot(q))

[직접]  213300
[CoT]
 3 × 79000 = 237000  
10% of 237000 = 23700  
237000 − 23700 = 213300  
정답: 213300


In [11]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

def verify(question: str, cot_answer: str) -> str:
    """제출한 풀이를 모델에게 다시 검산시켜 신뢰도를 높인다."""
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"[문제] {question}\n[제출한 풀이]\n{cot_answer}\n\n"
                    "위 풀이가 맞는지 다시 계산해 검산하라. "
                    "틀렸다면 올바른 값을, 맞다면 그대로 '정답: <숫자>'로 확정하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [12]:
q = "이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은?"
first = ask_cot(q)          # 1차 풀이
print("[검산 결과]\n", verify(q, first))   # 2차 검산

[검산 결과]
 정답: 213300


In [13]:
import os
import pathlib
import re
import pandas as pd
from openai import OpenAI

# OpenAI 클라이언트 초기화 및 모델 설정
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

DATA_PATH = pathlib.Path("./data")
# math_word_problems.csv = 쇼핑 계산 문제 8개. answer 컬럼이 정수 정답.
df = pd.read_csv(DATA_PATH / "math_word_problems.csv")

print("문제 수:", len(df))
print(df.iloc[0]["question"], "→ 정답:", df.iloc[0]["answer"])

문제 수: 8
승승장구몰에서 이어버드를 79000원에 3개 샀는데 10% 쿠폰을 받았습니다. 총 결제액은? → 정답: 213300


In [14]:
def extract_number(text: str):
    """답변에서 마지막 숫자를 정수로 반환(콤마 제거)."""
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    return int(nums[-1].replace(",", "")) if nums else None

def ask_direct(question: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 최종 숫자(원)만 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

def ask_cot(question: str) -> str:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": (
                    f"{question}\n"
                    "단계적으로 풀어라. 각 계산을 한 줄씩 쓰고, "
                    "맨 마지막 줄에 '정답: <숫자>' 형식으로 답하라."
                )
            }
        ],
        temperature=0,
    )
    return resp.choices[0].message.content

In [15]:
direct_ok = cot_ok = 0
for _, row in df.iterrows():
    ans = int(row["answer"])
    d = extract_number(ask_direct(row["question"]))   # 직접 답변의 숫자
    c = extract_number(ask_cot(row["question"]))      # CoT 답변의 숫자
    direct_ok += (d == ans)     # bool(True/False)은 1/0으로 더해진다
    cot_ok += (c == ans)
    print(f"{row['problem_id']} 정답={ans:>7} | 직접={d} {'O' if d==ans else 'X'}"
          f" | CoT={c} {'O' if c==ans else 'X'}")

n = len(df)
print(f"\n직접 답변 정답률 : {direct_ok}/{n} = {direct_ok/n:.0%}")
print(f"CoT  정답률      : {cot_ok}/{n} = {cot_ok/n:.0%}")

M01 정답= 213300 | 직접=213300 O | CoT=213300 O
M02 정답= 386650 | 직접=386650 O | CoT=386650 O
M03 정답= 107000 | 직접=107000 O | CoT=107000 O
M04 정답=  53000 | 직접=53000 O | CoT=53000 O
M05 정답= 128800 | 직접=128800 O | CoT=128800 O
M06 정답=   5320 | 직접=5320 O | CoT=5320 O
M07 정답=  94400 | 직접=94400 O | CoT=94400 O
M08 정답= 351000 | 직접=351000 O | CoT=351000 O

직접 답변 정답률 : 8/8 = 100%
CoT  정답률      : 8/8 = 100%


In [18]:
import os
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

def ask_direct(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n설명 없이 정답만 한 단어로 답하라."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens

def ask_cot(question: str):
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "user", "content": f"{question}\n단계적으로 풀어라. 마지막 줄에 '정답: <값>'."}
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip(), resp.usage.total_tokens


In [19]:
cases = [
    ("배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?", "50"),
    ("5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?", "5"),  # 정답 5분
]

for q, gold in cases:
    d_txt, _ = ask_direct(q)
    c_txt, _ = ask_cot(q)
    print(f"Q: {q}")
    print(f"  정답: {gold} / 직접: {d_txt[:20]} / CoT 마지막줄: {c_txt.splitlines()[-1]}")

Q: 배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?
  정답: 50 / 직접: 50원 / CoT 마지막줄: 정답: 50원
Q: 5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?
  정답: 5 / 직접: 5 / CoT 마지막줄: **정답: 5**


In [20]:
####################################################
# 토큰 합산용 변수 초기화
direct_tok_sum = 0
cot_tok_sum = 0
cases = [
    ("배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?", "50"),
    ("5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?", "5"),  # 정답 5분
]

for q, gold in cases:
    d_txt, d_tok = ask_direct(q)
    c_txt, c_tok = ask_cot(q)
    
    # 토큰 수 누적
    direct_tok_sum += d_tok
    cot_tok_sum += c_tok
    
    last_line_cot = c_txt.splitlines()[-1] if c_txt.splitlines() else c_txt
    print(f"Q: {q}")
    print(f"  정답: {gold} / 직접: {d_txt[:20]} / CoT 마지막줄: {last_line_cot}")

print("-" * 50)


Q: 배트와 공이 합쳐서 1,100원. 배트는 공보다 1,000원 비싸다. 공은?
  정답: 50 / 직접: 50원 / CoT 마지막줄: 정답: 50원
Q: 5대 기계가 5개를 5분에 만든다. 100대가 100개를 만드는 데 몇 분?
  정답: 5 / 직접: 5 / CoT 마지막줄: **정답: 5**
--------------------------------------------------


In [21]:
# (전체 케이스의 토큰을 합산)
print(f"총 토큰 — 직접: {direct_tok_sum} / CoT: {cot_tok_sum} "
      f"(CoT가 {cot_tok_sum - direct_tok_sum}토큰 더 사용)")

총 토큰 — 직접: 899 / CoT: 1101 (CoT가 202토큰 더 사용)


In [ ]:
import os
import pathlib
import re
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

DATA_PATH = pathlib.Path("./data")
df = pd.read_csv(DATA_PATH / "math_word_problems.csv")

def extract_number(text):
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    return int(nums[-1].replace(",", "")) if nums else None

def ask_cot(q):
    r = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": f"{q}\n단계적으로 풀어라. 각 계산을 한 줄씩 쓰고, 맨 마지막 줄에 '정답: <숫자>'."
            } 
        ],
        temperature=0
    )
    return r.choices[0].message.content

def ask_nostep(q):   # "단계적으로 풀어라" 문구만 제거 (형식은 동일)
    r = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                "role": "user",
                "content": f"{q}\n각 계산을 한 줄씩 쓰고, 맨 마지막 줄에 '정답: <숫자>'."
            }
        ],
        temperature=0
    )
    return r.choices[0].message.content

cot_ok = nostep_ok = 0
for _, row in df.iterrows():
    ans = int(row["answer"])
    cot_ok    += (extract_number(ask_cot(row["question"]))    == ans)
    nostep_ok += (extract_number(ask_nostep(row["question"])) == ans)

n = len(df)
print(f"CoT 정답률: {cot_ok}/{n} | 단계제거 정답률: {nostep_ok}/{n}")

CoT 정답률: 8/8 | 단계제거 정답률: 7/8


In [17]:
import os
from dotenv import load_dotenv
import pathlib
import re
from collections import Counter
import pandas as pd
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("nvidiaapi_key"),
)
OPENAI_MODEL = "openai/gpt-oss-20b"

DATA_PATH = pathlib.Path("./data")
df = pd.read_csv(DATA_PATH / "math_word_problems.csv")

def extract_number(text):
    nums = re.findall(r"-?\d[\d,]*", text.replace(" ", ""))
    return int(nums[-1].replace(",", "")) if nums else None

def self_consistency(question: str, n: int = 3):
    """같은 문제를 n번 풀어 가장 많이 나온 답(다수결)을 채택한다."""
    answers = []
    for _ in range(n):
        r = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": f"{question}\n단계적으로 풀고 마지막 줄에 '정답: <숫자>'."
                }
            ],
            temperature=0.7,  # 다양성 위해 약간 높임
        )
        answers.append(extract_number(r.choices[0].message.content))
    print("개별 답:", answers)
    winner = Counter(answers).most_common(1)[0][0]   # 최빈값
    print("다수결 답:", winner)
    return winner

if __name__ == "__main__":
    row = df.iloc[0]
    result = self_consistency(row["question"], n=3)
    print("실제 정답:", int(row["answer"]), "→", "맞음" if result == int(row["answer"]) else "틀림")

개별 답: [213300, 213300, 213300]
다수결 답: 213300
실제 정답: 213300 → 맞음
